# The gain of a question

A question splits a parent into a left child and a right child. The gain is how much impurity disappears. With Gini,

$$
\operatorname{Gain}
= G(\text{parent})
- \left(
\frac{n_L}{n} G(\text{left})
+ \frac{n_R}{n} G(\text{right})
\right)
$$

The weights $n_L/n$ and $n_R/n$ stop a tiny pure leaf from pretending it cleaned the whole parent.

There are two candidate questions that use a midpoint between training values. Lengths are $1$, $2$, and $5$, so the length cuts are $3/2$ and $7/2$. Weights are $1$, $2$, and $4$, so the weight cuts are $3/2$ and $3$.


## Length at most $7/2$

Left: the six parts with length $1$ or $2$. Four pass and two fail.

$$
G_L = 1 - \left(\frac{4}{6}\right)^2 - \left(\frac{2}{6}\right)^2
= 1 - \frac{16}{36} - \frac{4}{36}
= \frac{16}{36}
= \frac{4}{9}
$$

Right: the two parts with length $5$, both fail. $G_R = 0$.

$$
\frac{6}{8}\cdot\frac{4}{9} + \frac{2}{8}\cdot 0 = \frac{1}{3}
$$

Parent Gini is $1/2$, so

$$
\operatorname{Gain} = \frac{1}{2} - \frac{1}{3} = \frac{1}{6}
$$

Weight at most $3$ does the same arithmetic with the groups swapped in role: six mixed parts on one side, two pure fails on the other. Its gain is also $1/6$.


## The weaker cuts

Length at most $3/2$ puts three parts on the left: two pass, one fail.

$$
G = 1 - \left(\frac{2}{3}\right)^2 - \left(\frac{1}{3}\right)^2 = \frac{4}{9}
$$

The right side has five parts: two pass, three fail.

$$
G = 1 - \left(\frac{2}{5}\right)^2 - \left(\frac{3}{5}\right)^2
= 1 - \frac{4}{25} - \frac{9}{25}
= \frac{12}{25}
$$

$$
\frac{3}{8}\cdot\frac{4}{9} = \frac{1}{6} = \frac{5}{30},
\qquad
\frac{5}{8}\cdot\frac{12}{25} = \frac{3}{10} = \frac{9}{30}
$$

The weighted impurity is $14/30 = 7/15$. Gain:

$$
\frac{1}{2} - \frac{7}{15} = \frac{15}{30} - \frac{14}{30} = \frac{1}{30}
$$

Weight at most $3/2$ has this same gain, $1/30$. Both are real improvements. Both lose to $1/6$.


In [1]:
# Every midpoint, scored by Gini gain. Two winners tie at 1/6.
from fractions import Fraction

parts = [
    (1, 1, "pass"), (1, 2, "pass"), (2, 1, "pass"), (2, 2, "pass"),
    (1, 4, "fail"), (2, 4, "fail"), (5, 1, "fail"), (5, 2, "fail"),
]

def gini(labels):
    n = len(labels)
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return 1 - sum(Fraction(c, n) ** 2 for c in counts.values())

def split_gain(feature, threshold):
    left = [row for row in parts if row[feature] <= threshold]
    right = [row for row in parts if row[feature] > threshold]
    parent = gini([row[2] for row in parts])
    weighted = (
        Fraction(len(left), 8) * gini([row[2] for row in left])
        + Fraction(len(right), 8) * gini([row[2] for row in right])
    )
    return parent - weighted

scores = {
    ("length", Fraction(3, 2)): split_gain(0, Fraction(3, 2)),
    ("length", Fraction(7, 2)): split_gain(0, Fraction(7, 2)),
    ("weight", Fraction(3, 2)): split_gain(1, Fraction(3, 2)),
    ("weight", 3): split_gain(1, 3),
}
assert scores[("length", Fraction(7, 2))] == Fraction(1, 6)
assert scores[("weight", 3)] == Fraction(1, 6)
assert scores[("length", Fraction(3, 2))] == Fraction(1, 30)
assert scores[("weight", Fraction(3, 2))] == Fraction(1, 30)
for key, value in scores.items():
    print(key, value)


('length', Fraction(3, 2)) 1/30
('length', Fraction(7, 2)) 1/6
('weight', Fraction(3, 2)) 1/30
('weight', 3) 1/6


## A pitfall

A gain of $1/6$ on two different questions is a tie, not a rounding accident. The next lesson's tree has to publish the rule that breaks it. Choosing silently makes a later "feature importance" depend on an unstated coin flip.

## What to carry forward

Gain is the parent's Gini minus the size-weighted Gini of the children. On these eight parts the best gain is $1/6$, shared by length $\le 7/2$ and weight $\le 3$. The other two midpoints gain only $1/30$.
